## Map SIF network nodes (UniProt and ChEBI identifiers) to Gene Names (for proteins) and Molecule Names (for molecules)

In [15]:
from SPARQLWrapper import SPARQLWrapper, TURTLE, JSON, CSV
import subprocess
import time
import os 
from requests.utils import requote_uri
from urllib.parse import quote
import re
import rdflib
import pandas as pd
from pathlib import Path

In [2]:
results_files = list()
for file in os.listdir("../../Results/ReactomeHomoSapiens95/Meaning3/QueryResults/"):
    print(file)
    results_files.append(file)

print(results_files)
aggregatedNetwork = pd.concat([pd.read_csv(f"../../Results/ReactomeHomoSapiens95/Meaning3/QueryResults/{file}", header=0, sep=",") for file in results_files]) 
print(aggregatedNetwork.head())
print(len(aggregatedNetwork))

04-ControlsExpressionOf.csv
11-ControlsTransportOfChemical.csv
08-NeighborOf.csv
03-ControlsPhosphorylationOf.csv
02-ControlsTransportOf.csv
14-UsedToProduce.csv
09-ConsumptionControledBy.csv
13-ReactsWith.csv
01-ControlsStateChangeOf.csv
06-InComplexWith.csv
['04-ControlsExpressionOf.csv', '11-ControlsTransportOfChemical.csv', '08-NeighborOf.csv', '03-ControlsPhosphorylationOf.csv', '02-ControlsTransportOf.csv', '14-UsedToProduce.csv', '09-ConsumptionControledBy.csv', '13-ReactsWith.csv', '01-ControlsStateChangeOf.csv', '06-InComplexWith.csv']
                           Source                       Interaction  \
0   reactome:ProteinReference9082  abstraction:ControlsExpressionOf   
1   reactome:ProteinReference4458  abstraction:ControlsExpressionOf   
2   reactome:ProteinReference6440  abstraction:ControlsExpressionOf   
3   reactome:ProteinReference2808  abstraction:ControlsExpressionOf   
4  reactome:ProteinReference10409  abstraction:ControlsExpressionOf   

                      

In [14]:
Reactome95EntitiesIDS = pd.read_csv(
    "../../Results/ReactomeHomoSapiens95/UtilityFiles/ReactomeHomoSapiens95EntityRefsIDs.csv",
    sep=",",
    header=0
)

Reactome95EntitiesIDS['ref'] = Reactome95EntitiesIDS.iloc[:, 0].str.replace(
    "http://www.reactome.org/biopax/95/48887#", "reactome:"
)

print(Reactome95EntitiesIDS.head())

dico_entity_ref = dict(zip(
    Reactome95EntitiesIDS['ref'],
    Reactome95EntitiesIDS.iloc[:, 2]
))

print(dico_entity_ref)
dico_entity_ref_names = dict(zip(
    Reactome95EntitiesIDS['ref'],
    Reactome95EntitiesIDS.iloc[:, 1]
))

print(dico_entity_ref_names)

                                           entityRef           entityRefName  \
0  http://www.reactome.org/biopax/95/48887#Protei...                   RING6   
1  http://www.reactome.org/biopax/95/48887#Protei...  UniProt:P28067 HLA-DMA   
2  http://www.reactome.org/biopax/95/48887#Protei...                     DMA   
3  http://www.reactome.org/biopax/95/48887#Protei...                 HLA-DMA   
4  http://www.reactome.org/biopax/95/48887#Protei...   UniProt:Q96SR6 ZNF382   

  entityID                             ref  
0   P28067   reactome:ProteinReference9243  
1   P28067   reactome:ProteinReference9243  
2   P28067   reactome:ProteinReference9243  
3   P28067   reactome:ProteinReference9243  
4   Q96SR6  reactome:ProteinReference10125  
{'reactome:ProteinReference9243': 'P28067', 'reactome:ProteinReference10125': 'Q96SR6', 'reactome:ProteinReference1115': 'Q9Y2X0', 'reactome:ProteinReference2198': 'Q86VD7', 'reactome:ProteinReference7163': 'Q9BYT8', 'reactome:ProteinReference4223':

In [ ]:
aggregatedNetworkIDS = pd.DataFrame({
    "Source": aggregatedNetwork.iloc[:, 0].map(dico_entity_ref),
    "Interaction": aggregatedNetwork.iloc[:, 1],
    "Target": aggregatedNetwork.iloc[:, 2].map(dico_entity_ref)
})

aggregatedNetworkIDS.to_csv("../../Results/ReactomeHomoSapiens95/Meaning3/SIF/Reactome95-SIF-Meaning3.tsv", sep="\t", header=0, index=False)
print(aggregatedNetworkIDS)

        Source                       Interaction    Target
0       Q8TAQ2  abstraction:ControlsExpressionOf    P14679
1       P62805  abstraction:ControlsExpressionOf    Q9HCL2
2       P00533  abstraction:ControlsExpressionOf    P01100
3       P51948  abstraction:ControlsExpressionOf    Q13315
4       Q86U70  abstraction:ControlsExpressionOf    Q8NGC3
...        ...                               ...       ...
314680  P24928         abstraction:InComplexWith    Q9C0J8
314681  P49792         abstraction:InComplexWith  P52948-3
314682  P31371         abstraction:InComplexWith    P42081
314683  Q9UPV0         abstraction:InComplexWith    P63167
314684  Q9P0N5         abstraction:InComplexWith    P51955

[866451 rows x 3 columns]


In [35]:
unique_entities = set(aggregatedNetworkIDS['Source']).union(
    set(aggregatedNetworkIDS['Target'])
)

SIF_entities = pd.DataFrame({'ID': list(unique_entities)})

output_path = Path("../../Results/ReactomeHomoSapiens95/UtilityFiles/Entities-Reactome95-SIF-Meaning3.txt")
SIF_entities.to_csv(output_path, header=None, index=False)
print(SIF_entities.head())

       ID
0  O15540
1  P02511
2  Q8WWZ3
3  O43674
4  Q15139


### Map Uniprot IDs to Gene Names

1. Create SIF node table with proteins represented with their uniprot ids
2. Go to https://www.uniprot.org/help/id_mapping and load SIF node table as text file and run mapping "UniProtKB AC/ID" to "UniProtKB/Swiss-Prot" (get back the mappings of the UniProt IDs to UniProt reviewed IDs associated to a unique Gene Name) + filter Homo Sapiens results
3. Download results as tsv file "../../Data/UniprotMapping/Reactome95-Meaning2-SIFProteinsReviewed.tsv"
4. Create dictionary of Uniprot IDs and Gene names

In [36]:
# load uniprot mapping file and sotre data in a dict
uniprot_mapping_sif_file = pd.read_table(
    "../../Data/Uniprot_mapping/Reactome95-Meaning3-SIFProteinsReviewed.tsv",
    sep="\t"
)

# map Uniprot reviewed IDs to gene names 
dico_SIF_uniprot_mapping = dict(
    zip(
        uniprot_mapping_sif_file.iloc[:, 0],
        uniprot_mapping_sif_file.iloc[:, 5]
    )
)

print(dico_SIF_uniprot_mapping)

{'A0A075B6P5': 'IGKV2-28', 'A0A075B6S6': 'IGKV2D-30', 'P01562': 'IFNA1', 'A0A096LP49': 'CCDC187', 'A0A096LP55': 'UQCRHL', 'A0A0A6YYK7': 'TRAV19', 'A0A0C4DH25': 'IGKV3D-20', 'A0A0C4DH73': 'IGKV1-12', 'A0A183': 'LCE6A C1orf44', 'A0A1W2PPF3': 'DUXB', 'A0AVF1': 'IFT56 TTC26', 'A0AVI4': 'TMEM129', 'A0AVK6': 'E2F8', 'A0AVT1': 'UBA6 MOP4 UBE1L2', 'A0FGR8': 'ESYT2 FAM62B KIAA1228', 'A0FGR9': 'ESYT3 FAM62C', 'A0JLT2': 'MED19 LCMR1', 'A0JNW5': 'BLTP3B KIAA0701 SHIP164 UHRF1BP1L', 'A0M8Q6': 'IGLC7', 'A0MZ66': 'SHTN1 KIAA1598', 'A0PJK1': 'SLC5A10 SGLT5', 'A0PJW6': 'TMEM223', 'A0PJZ3': 'GXYLT2 GLT8D4', 'A1A4S6': 'ARHGAP10 GRAF2', 'A1A5B4': 'ANO9 PIG5 TMEM16J TP53I5', 'A1L188': 'NDUFAF8 C17orf89', 'A1L190': 'SYCE3 C22orf41 THEG2', 'A1L390': 'PLEKHG3 KIAA0599', 'A1L3X0': 'ELOVL7', 'A2NJV5': 'IGKV2-29', 'A2RRD8': 'ZNF320', 'A2RRP1': 'NBAS NAG', 'A2RU49': 'HYKK AGPHD1', 'A2RUC4': 'TYW5 C2orf60', 'A2RUS2': 'DENND3 KIAA0870', 'A2VDF0': 'FUOM C10orf125', 'A2VEC9': 'SSPOP KIAA2036 SSPO', 'A3KFT3': 'OR2M5 O

### Map ChEBI IDs to molecule names

1. get back mapping file from https://www.ebi.ac.uk/chebi/downloads
2. Create dictionary of ChEBI identifiers and associated molecule names

In [37]:
chebi_mapping_file = pd.read_table("../../Data/Chebi_mapping/names.tsv")
print(chebi_mapping_file.head())

      id  compound_id                                name        type  \
0      8            3        ((R)-3-Hydroxybutanoyl)(n-2)     SYNONYM   
1  61706            7                     1α,6α-car-3-ene  IUPAC NAME   
2  61701            7            (1<i>S</i>)-(+)-3-carene     SYNONYM   
3  61702            7             (<i>S</i>)-(+)-3-carene     SYNONYM   
4  61703            7  (1<i>S</i>,6<i>R</i>)-(+)-3-carene     SYNONYM   

   status_id  adapted language_code                    ascii_name  
0          3    False            en  ((R)-3-Hydroxybutanoyl)(n-2)  
1          1    False            en       1alpha,6alpha-car-3-ene  
2          1    False            en             (1S)-(+)-3-carene  
3          1    False            en              (S)-(+)-3-carene  
4          1    False            en          (1S,6R)-(+)-3-carene  


In [38]:
unique_entries = chebi_mapping_file.drop_duplicates(subset=["id", "ascii_name"])

dico_SIF_chebi_mapping = {
    f"CHEBI:{row[1]}": row[7]
    for _, row in unique_entries.iterrows()
}

print({k: dico_SIF_chebi_mapping[k] for k in list(dico_SIF_chebi_mapping.keys())[:5]})

/tmp/ipykernel_104226/650446419.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  f"CHEBI:{row[1]}": row[7]


{'CHEBI:3': '((R)-3-Hydroxybutanoyl)(n-2)', 'CHEBI:7': '(+)-alpha-carene', 'CHEBI:8': '(+)-hydroxycalamenene', 'CHEBI:9': '(+)-Adlumine', 'CHEBI:10': '(+)-Atherospermoline'}


### Create new SIF network with Gene names and molecule names


In [40]:
def map_entity(entity, uniprot_dict, chebi_dict):
    if entity in uniprot_dict:
        return uniprot_dict[entity]
    elif entity in chebi_dict:
        return chebi_dict[entity]
    return entity

aggregatedNetworkGeneNames = aggregatedNetworkIDS.copy()
aggregatedNetworkGeneNames['Source'] = aggregatedNetworkIDS['Source'].apply(
    lambda x: map_entity(x, dico_SIF_uniprot_mapping, dico_SIF_chebi_mapping)
)
aggregatedNetworkGeneNames['Target'] = aggregatedNetworkIDS['Target'].apply(
    lambda x: map_entity(x, dico_SIF_uniprot_mapping, dico_SIF_chebi_mapping)
)

unmapped_entities = set()
for col in ['Source', 'Target']:
    unmapped = aggregatedNetworkGeneNames[aggregatedNetworkGeneNames[col] == aggregatedNetworkIDS[col]][col]
    unmapped_entities.update(unmapped.tolist())

print("Entités non mappées:", unmapped_entities)
print(len(unmapped_entities))

output_path = Path("../../Results/ReactomeHomoSapiens95/Meaning3/SIF/Reactome95-SIF-Meaning3-Names.tsv")
aggregatedNetworkGeneNames.to_csv(
    output_path,
    sep="\t",
    header=False,
    index=False
)

Entités non mappées: {'P0AE06', 'P62593', '7447', 'Q04875', 'M4Q6L3', 'F5HA10', 'Q9F663', '11237', '5585', 'F5H9N9', 'Q98325', 'P61764-1', 'Q6SW70', '3902', '5046', 'CHEBI:195408', 'P12491', 'P37313', 'CHEBI:180684', '6789', 'Q76RF1', 'P9WN25', 'F5HE05', 'F5HE12', 'O53692', 'Q04878', 'Q6SW87', 'P0ABU7', 'Q5NV91', 'Q6SVX2', 'F5HET1', 'P69723', 'P41785', 'F5HGQ8', 'F5HDK1', 'CHEBI:139352', 'CHEBI:21507', 'CHEBI:81621', 'Q6SW84', 'P09986', 'Q6SW66', 'P04150-9', 'P03377', 'P05549-1', 'Q13976-1', 'Q8VNV4', 'A2NXD2', 'P04581', 'Q9BY41-1', 'P04608', 'A2KUC3', 'F5HEA3', '8967', 'F5HC14', '8966', 'P03468', 'P62993-1', 'P03508', 'Q8XAL7', 'P9WGE9', 'Q04874', 'P23847', 'P29353-1', '10090', 'F5HGI9', 'P21731-3', 'O94956-1', 'O15519-1', 'Q9Y5V3-2', 'P9WL31', 'Q96RI1-1', 'F5HB41', 'F5HBR4', 'Q6SW65', 'P37231-1', 'P04580', 'P08235-3', '5082', 'Q5NV66', 'Q6SW04', 'F5HFG3', 'P18640', 'P02980', 'Q6SWC7', 'P15031', 'P0A937', 'CHEBI:195410', 'P12931-1', 'Q03001-3', 'F5HGU6', 'Q5NV89', 'Q96RI1-3', 'P04601'